# Ch.4 — Support Vector Machines for Smiling Detection

**FaceAI**: Maximum-margin classification with linear and RBF kernels.

**Goal**: Push accuracy from 88% (LogReg) to ~89% (SVM) via margin maximization.

**Key concepts**: Hyperplane, margin, support vectors, kernel trick, C/gamma trade-off.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import numpy, matplotlib.pyplot, Path
# 2. Import make_classification, train_test_split, cross_val_score
# 3. Import StandardScaler; import SVC, LinearSVC from sklearn.svm
# 4. Import LogisticRegression; import make_pipeline from sklearn.pipeline
# 5. Import classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
# 6. Import PCA from sklearn.decomposition
# 7. Set IMG_DIR, SAVE_KW, np.random.seed(42)
#
# Hint:
#   from sklearn.svm import SVC, LinearSVC
#   from sklearn.decomposition import PCA
#   from sklearn.pipeline import make_pipeline
#   np.random.seed(42)


## §0 Data — CelebA Smiling (same setup)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use make_classification() with same parameters as Ch.1/Ch.2
#    (n_samples=5000, n_features=200, n_informative=40, random_state=42)
# 2. Split with train_test_split(test_size=0.2, stratify=y, random_state=42)
# 3. Scale with StandardScaler (fit_transform on train, transform on test)
# 4. Print shapes
#
# Hint:
#   X, y = make_classification(n_samples=5000, n_features=200, n_informative=40,
#                              n_redundant=20, n_clusters_per_class=3,
#                              weights=[0.52, 0.48], flip_y=0.05, random_state=42)
#   scaler    = StandardScaler()
#   X_train_s = scaler.fit_transform(???)
#   X_test_s  = scaler.transform(???)


## §1 Linear SVM vs Logistic Regression

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Train LogisticRegression(C=1.0, max_iter=500, random_state=42) as baseline
# 2. Train SVC(kernel='linear', C=1.0, probability=True, random_state=42)
# 3. Print accuracy for both
# 4. Print support vector count: linear_svm.n_support_ (array of counts per class)
#
# Hint:
#   lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
#   lr.fit(X_train_s, y_train)
#   linear_svm = SVC(kernel='linear', C=???, probability=True, random_state=42)
#   linear_svm.fit(X_train_s, y_train)
#   print(f"Support vectors: {linear_svm.n_support_}  ({sum(linear_svm.n_support_)/len(y_train)*100:.1f}%)")


## §2 RBF Kernel — Non-linear Boundaries

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Train SVC(kernel='rbf', C=10, gamma=0.01, probability=True, random_state=42)
# 2. Get y_pred = rbf_svm.predict(X_test_s)
# 3. Get y_prob = rbf_svm.predict_proba(X_test_s)[:, 1]
# 4. Print accuracy, ROC-AUC (roc_auc_score), support vector count, and
#    full classification_report
#
# Hint:
#   rbf_svm = SVC(kernel='rbf', C=???, gamma=???, probability=True, random_state=42)
#   rbf_svm.fit(X_train_s, y_train)
#   y_prob = rbf_svm.predict_proba(???)[: , 1]
#   print(f"Support vectors: {sum(rbf_svm.n_support_)}")


## §3 C and Gamma Exploration

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define C_values=[0.1, 1, 10, 100] and gamma_values=[0.001, 0.01, 0.1, 1.0]
# 2. Use a 2000-sample subset: X_sub = X_train_s[:2000], y_sub = y_train[:2000]
# 3. For each (C, gamma) pair run 3-fold CV; store mean accuracy in scores[i, j]
# 4. Visualise as ax.imshow(scores, cmap='YlOrRd'); annotate each cell with value
# 5. Save to IMG_DIR / 'c_gamma_heatmap.png'
#
# Hint:
#   scores = np.zeros((len(C_values), len(gamma_values)))
#   for i, C in enumerate(C_values):
#       for j, gamma in enumerate(gamma_values):
#           cv_scores = cross_val_score(SVC(kernel='rbf', C=C, gamma=gamma,
#                                          random_state=42), X_sub, y_sub, cv=3)
#           scores[i, j] = cv_scores.mean()
#   ax.imshow(scores, cmap='YlOrRd', aspect='auto')


## §4 Margin Visualization (2D PCA Projection)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Reduce X_train_s[:1000] to 2D using PCA(n_components=2)
# 2. Train SVC(kernel='rbf', C=10, gamma=0.5) on the 2D data
# 3. Build a meshgrid covering the PCA space; call svm_2d.predict() to fill Z
# 4. Plot ax.contourf(xx, yy, Z, alpha=0.3); scatter data points
# 5. Highlight support vectors (svm_2d.support_ gives indices) with larger circles
# 6. Save to IMG_DIR / 'svm_decision_boundary.png'
#
# Hint:
#   pca = PCA(n_components=2)
#   X_pca = pca.fit_transform(X_train_s[:1000])
#   svm_2d = SVC(kernel='rbf', C=???, gamma=???, random_state=42)
#   xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
#   Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
#   sv = svm_2d.support_          # indices of support vectors in the training set


## §5 ROC Comparison: LogReg vs SVM

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Build models_cmp dict: {'LogReg': lr, 'Linear SVM': linear_svm, 'RBF SVM': rbf_svm}
# 2. For each model call predict_proba(X_test_s)[:, 1] to get probabilities
# 3. Compute fpr, tpr = roc_curve(y_test, y_p) and auc = roc_auc_score(...)
# 4. Plot all three curves on one axes; add AUC to each label
# 5. Save to IMG_DIR / 'roc_comparison.png'
#
# Hint:
#   models_cmp = {'LogReg': lr, 'Linear SVM': linear_svm, 'RBF SVM': rbf_svm}
#   for name, model in models_cmp.items():
#       y_p = model.predict_proba(???)[: , 1]
#       fpr, tpr, _ = roc_curve(y_test, y_p)
#       auc = roc_auc_score(???, y_p)
#       ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')


## §6 Support Vector Analysis

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Get n_sv = rbf_svm.n_support_  (array: [Not-Smiling SVs, Smiling SVs])
# 2. Compute total_sv = sum(n_sv); non-SVs = len(y_train) - total_sv
# 3. Create a pie chart with 3 slices: Not-Smiling SVs, Smiling SVs, Non-SVs
# 4. Print percentage of training data that are support vectors
# 5. Save to IMG_DIR / 'support_vectors_pie.png'
#
# Hint:
#   n_sv   = rbf_svm.n_support_
#   sizes  = [n_sv[0], n_sv[1], len(y_train) - sum(n_sv)]
#   labels = ['Not Smiling SVs', 'Smiling SVs', 'Non-SVs']
#   ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)


## §7 Summary

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Print separator line and chapter title
# 2. Print accuracy for lr, linear_svm, and rbf_svm
# 3. Print support vector count and ACCURACY constraint status (~89%)
# 4. Print note pointing to Ch.5 for systematic tuning to 92%
#
# Hint:
#   print(f"  LogReg:     {lr.score(X_test_s, y_test):.3f}")
#   print(f"  Linear SVM: {linear_svm.score(X_test_s, y_test):.3f}")
#   print(f"  RBF SVM:    {rbf_svm.score(X_test_s, y_test):.3f}")
#   print(f"Support vectors: {sum(rbf_svm.n_support_)} ({sum(rbf_svm.n_support_)/len(y_train)*100:.1f}%)")


## Exercises

1. **Kernel comparison**: Train SVM with linear, poly (degree 2, 3), and RBF kernels. Compare accuracy and training time.
2. **class_weight for Bald**: Create imbalanced Bald dataset (2.5%), train SVM with/without `class_weight='balanced'`. Compare F1.
3. **Scaling impact**: Train RBF SVM without StandardScaler. How much does accuracy drop?

In [ ]:
# Exercise 1: Kernel comparison
# TODO: Implement this cell
#
# Hint: for kernel, degree in [('linear', 3), ('poly', 2), ('poly', 3), ('rbf', 3)]:
#   use SVC(kernel=kernel, degree=degree); record test accuracy and time.time() duration


In [ ]:
# Exercise 2: class_weight for Bald
# TODO: Implement this cell
#
# Hint: create imbalanced data with make_classification(weights=[0.975, 0.025]);
#   compare SVC(class_weight=None) vs SVC(class_weight='balanced') using f1_score


In [ ]:
# Exercise 3: Scaling impact
# TODO: Implement this cell
#
# Hint: train SVC(kernel='rbf', C=10, gamma=0.01) on raw X_train (no scaling)
#   vs X_train_s (scaled); compare test accuracy — RBF is highly sensitive to scale
